# Цель данного документа

Целью данного документа является демонстрация примеров из главы документации Datapipe [[In progress] Как устроена работа с индексами и метаданными](https://www.notion.so/epoch8/In-progress-b77d3fb25f5c4f74a2ec41007972ddab?source=copy_link).

# Подготовка

## Настройки проекта

Настройки проекта:
* `PROJECT_LOCATION` - локация проекта, где будут созданы сырые и обработанные данные, описаны зависимости Python, а также сохранены метаданные Datapipe в виде SQLite базы данных;
* `RAW_DATA_LOCATION` - поддиректория для сырых данных;
* `PROCESSED_DATA_LOCATION` - поддиректория для обработанных данных;
* `RAW_DATA` - сырые данные, над которыми будут производиться операции из примеров;
* `FILES_TO_CLEAN` - список файлов для очистки, что позволит повторно запустить примеры.

In [1]:
PROJECT_LOCATION = "/content/"
RAW_DATA_LOCATION = "data/raw/cars/"
PROCESSED_DATA_LOCATION = "data/processed/cars/"

RAW_DATA = {
    "record_1.json": '{"model_id": 1, "manufacture_country": "Germany", "color": "Red", "price": 15000, "year": 2023, "new": false}',
    "record_2.json": '{"model_id": 1, "manufacture_country": "Germany", "color": "Blue", "price": 20000, "year": 2026, "new": true}',
    "record_3.json": '{"model_id": 3, "manufacture_country": "Germany", "color": "Blue", "price": 25000, "year": 2026, "new": true}',
    "record_4.json": '{"model_id": 4, "manufacture_country": "Japan", "color": "Red", "price": 20000, "year": 2024, "new": false}',
    "record_5.json": '{"model_id": 5, "manufacture_country": "Japan", "color": "Blue", "price": 15000, "year": 2025, "new": true}',
}

FILES_TO_CLEAN = [
    "requirements.txt",
    "store.sqlite",
    "store.sqlite-shm",
    "store.sqlite-wal",
]

## Установка пакетов Python

При установке пакетов может потребоваться перезапуск ноутбука.

In [2]:
REQUIREMENTS = """
cityhash==0.4.10 ; python_version == "3.12"
click==8.3.2 ; python_version == "3.12"
cloudpickle==3.1.2 ; python_version == "3.12"
colorama==0.4.6 ; python_version == "3.12" and platform_system == "Windows"
datapipe-core==0.14.8 ; python_version == "3.12"
fsspec==2026.3.0 ; python_version == "3.12"
greenlet==3.4.0 ; (platform_machine == "aarch64" or platform_machine == "ppc64le" or platform_machine == "x86_64" or platform_machine == "amd64" or platform_machine == "AMD64" or platform_machine == "win32" or platform_machine == "WIN32") and python_version == "3.12"
importlib-metadata==8.7.1 ; python_version == "3.12"
markdown-it-py==4.0.0 ; python_version == "3.12"
mdurl==0.1.2 ; python_version == "3.12"
numpy==2.4.4 ; python_version == "3.12"
opentelemetry-api==1.41.0 ; python_version == "3.12"
opentelemetry-instrumentation-sqlalchemy==0.62b0 ; python_version == "3.12"
opentelemetry-instrumentation==0.62b0 ; python_version == "3.12"
opentelemetry-sdk==1.41.0 ; python_version == "3.12"
opentelemetry-semantic-conventions==0.62b0 ; python_version == "3.12"
packaging==26.1 ; python_version == "3.12"
pandas==3.0.2 ; python_version == "3.12"
pillow==11.3.0 ; python_version == "3.12"
psycopg2-binary==2.9.11 ; python_version == "3.12"
pygments==2.20.0 ; python_version == "3.12"
pysqlite3-binary==0.5.4.post2 ; python_version == "3.12"
python-dateutil==2.9.0.post0 ; python_version == "3.12"
pyyaml==6.0.3 ; python_version == "3.12"
rich==15.0.0 ; python_version == "3.12"
six==1.17.0 ; python_version == "3.12"
sqlalchemy-pysqlite3-binary==0.0.4 ; python_version == "3.12"
sqlalchemy==2.0.49 ; python_version == "3.12"
tqdm-loggable==0.2 ; python_version == "3.12"
tqdm==4.67.3 ; python_version == "3.12"
traceback-with-variables==2.2.1 ; python_version == "3.12"
typing-extensions==4.15.0 ; python_version == "3.12"
tzdata==2026.1 ; (sys_platform == "win32" or sys_platform == "emscripten") and python_version == "3.12"
wrapt==2.1.2 ; python_version == "3.12"
zipp==3.23.1 ; python_version == "3.12"
"""

In [3]:
with open(f"{PROJECT_LOCATION}requirements.txt", "w") as file:
  file.write(REQUIREMENTS)

In [4]:
! pip install -r "{PROJECT_LOCATION}requirements.txt"

Ignoring colorama: markers 'python_version == "3.12" and platform_system == "Windows"' don't match your environment
Ignoring tzdata: markers '(sys_platform == "win32" or sys_platform == "emscripten") and python_version == "3.12"' don't match your environment


## Импорты

In [5]:
from sqlalchemy import Column
from sqlalchemy import String
from sqlalchemy import Integer
from sqlalchemy import create_engine
from sqlalchemy import inspect

from datapipe.compute import Catalog
from datapipe.compute import DatapipeApp
from datapipe.compute import Pipeline
from datapipe.compute import Table
from datapipe.executor import ExecutorConfig
from datapipe.datatable import DataStore
from datapipe.step.batch_transform import BatchTransform
from datapipe.store.database import DBConn
from datapipe.store.pandas import TableStoreJsonLine

import os
import re
import time
import json
from pathlib import Path
from typing import List, Optional
import shutil

import fsspec
import pandas as pd
from datapipe.compute import ComputeStep, PipelineStep
from datapipe.step.datatable_transform import DatatableTransformStep
from datapipe.datatable import DataTable
from datapipe.run_config import RunConfig
from datapipe.compute import run_steps
from datapipe.store.filedir import _pattern_to_attrnames
from datapipe.store.filedir import _pattern_to_glob
from datapipe.store.filedir import _pattern_to_match
from datapipe.store.filedir import _pattern_to_patterns_or
from datapipe.types import Labels

import warnings
from sqlalchemy.exc import SAWarning, SADeprecationWarning

## Настройки пакетов Python

Настройки отображения Pandas: вывод данных на экран без ограничений по ширине и количеству столбцов.

In [6]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

Отключение некритичных предупреждений библиотеки `sqlalchemy` для работы с SQLite.

In [7]:
warnings.filterwarnings('ignore', category=SAWarning)
warnings.filterwarnings('ignore', category=SADeprecationWarning)

## Подготовка сырых данных

Запись сырых данных в директорию проекта, а также создание необходимых поддиректорий.

In [8]:
os.makedirs(f"{PROJECT_LOCATION}data/raw/cars", exist_ok=True)
os.makedirs(f"{PROJECT_LOCATION}data/processed/cars", exist_ok=True)

for file_name, data in RAW_DATA.items():
  with open(f"{PROJECT_LOCATION}{RAW_DATA_LOCATION}{file_name}", "w") as file:
    file.write(data)

## Функция сканирования входных данных

Входной точкой в пайплайн обработки данных для сырых данных является функция сканирования, которая сгенерирует мету для всех обнаруженных файлов в заданной директории. Таким образом, функция сканирования позволяет обнаружать новые сырые данные.

Привер выполнения функции будет рассмотрен далее, а принцип работы функции предлагается пропустить на данном этапе.

In [9]:
FILES_SCHEMA = [
    Column("filepath", String()),
    Column("size_bytes", Integer()),
]


class ScanFileList(PipelineStep):
    def __init__(
        self,
        filename_pattern: str,
        filename_output: str,
        output: str,
        labels: Optional[Labels] = None,
    ):
        self.output = output
        self.filename_pattern = filename_pattern
        self.filename_output = filename_output
        self.labels = labels

    def build_compute(self, ds: DataStore, catalog: Catalog) -> List[ComputeStep]:
        attrnames = _pattern_to_attrnames(self.filename_pattern)

        catalog.add_datatable(
            name=self.output,
            dt=Table(
                store=TableStoreJsonLine(
                    filename=self.filename_output.format(data=self.output),
                    primary_schema=[
                        Column(attrname, String(), primary_key=True)
                        for attrname in attrnames
                    ]
                    + FILES_SCHEMA,
                )
            ),
        )

        output_table = catalog.get_datatable(ds, self.output)

        return [
            DatatableTransformStep(
                name=f"scan_file_list__{self.output}",
                input_dts=[],
                output_dts=[output_table],
                func=scan_file_list,
                kwargs={"filename_pattern": self.filename_pattern},
                labels=self.labels,
            )
        ]


def scan_file_list(
    ds: DataStore,
    input_dts: List[DataTable],
    output_dts: List[DataTable],
    run_config: Optional[RunConfig],
    kwargs: dict,
) -> None:
    [output_dt] = output_dts
    now = time.time()

    filename_pattern = kwargs["filename_pattern"]

    protocol, path = fsspec.core.split_protocol(filename_pattern)

    if protocol is None or protocol == "file":
        filename_pattern = str(Path(path).resolve())
        filename_pattern_for_match = filename_pattern
        protocol_str = "" if protocol is None else "file://"
    else:
        filename_pattern = str(filename_pattern)
        filename_pattern_for_match = path
        protocol_str = f"{protocol}://"

    filename_patterns = _pattern_to_patterns_or(filename_pattern)
    attrnames = _pattern_to_attrnames(filename_pattern)
    filename_glob = [_pattern_to_glob(pat) for pat in filename_patterns]
    filename_match = _pattern_to_match(filename_pattern_for_match)

    files = fsspec.open_files(filename_glob)

    res = []

    for f in files:
        item = {
            "filepath": f"{protocol_str}{os.path.relpath(f.path)}",
            "size_bytes": files.fs.size(f.path),
        }

        m = re.match(filename_match, f.path)

        assert m is not None
        for attrname in attrnames:
            item[attrname] = m.group(attrname)

        res.append(item)

    output_dt.store_chunk(pd.DataFrame(res), now=now, run_config=run_config)
    output_dt.delete_stale_by_process_ts(now, now=now, run_config=run_config)

## Создание SQLite БД для меты

Для описания данных, а также для отслеживания изменений и рассчёта индекса транформации требуется реляционная БД. В данном примере используется локальный инстанс SQLite.

In [10]:
try:
    import pysqlite3
    sqla_engine = "sqlite+pysqlite3"
except ImportError:
    sqla_engine = "sqlite"

dbconn = DBConn(f"{sqla_engine}:///{PROJECT_LOCATION}store.sqlite")
ds = DataStore(dbconn)

# Шаг 1. Сканирование данных

В качестве первого шага пайплайна будет использоваться функция сканирования сырых данных, которая реализует точку входа в систему.

## Описание пайплайна

Для обнаружения и чтения сырых данных, а также для сохранения обработанных данных требуются шаблоны путей к файлам:
* `FILEPATH__RAW__CARS` - шаблон пути к сырым данным. В директории проекта и в поддиректории сырых данных будет произведён поиск файлов согласно шаблону `{{file_name}}.json`. Все файлы с расширением `json` будут учтены;
* `FILEPATH__PROCESSED__CARS` - аналогичный шаблон, но для записи обработанных данных.

In [11]:
FILEPATH__RAW__CARS = f"{PROJECT_LOCATION}{RAW_DATA_LOCATION}{{file_name}}.json"
FILEPATH__PROCESSED__CARS = f"{PROJECT_LOCATION}{PROCESSED_DATA_LOCATION}{{data}}.jsonl"

In [12]:
print(f"Фактический шаблон пути к сырым данным:\n{FILEPATH__RAW__CARS}")

Фактический шаблон пути к сырым данным:
/content/data/raw/cars/{file_name}.json


In [13]:
print(f"Фактический шаблон пути к обработанным данным:\n{FILEPATH__PROCESSED__CARS}")

Фактический шаблон пути к обработанным данным:
/content/data/processed/cars/{data}.jsonl


Функция сканирования самостоятельно генерирует каталог для описания сырых данных, поэтому описание каталога данных в коде не требуется на данном шаге.

Каталог данных содержит описание всех данных в системе и представляет собой набор таблиц в реляционной СУБД. В данном примере используется SQLite БД, где будут созданы таблицы, описывающие данные - таблицы метаданных.

In [14]:
catalog = Catalog({})

Пайплайн будет состоять из одной функции сканирования, в которую передаются следующие аргументы:
* `filename_pattern` - шаблон пути для поиска сырых данных;
* `filename_output` - шаблон пути для записи информации об отсканированных данных;
* `output` - именование данных на выходе из функции сканирования для каталога данных;
* `labels` - метки для разметки шагов трансформаций. Задаются пользователем и позволяют запускать шаги с определёнными метками.

In [15]:
pipeline = Pipeline(
    [
        ScanFileList(
            filename_pattern=FILEPATH__RAW__CARS,
            filename_output=FILEPATH__PROCESSED__CARS,
            output="cars_scanned",
            labels=[
                ("entity", "cars"),
                ("layer", "scan"),
                ("environment", "prod"),
            ],
        ),
    ]
)

Для инициализации пайплайна используются следующие аргументы:
* `ds` - Data Store, хранилище метаданных, реляционная СУБД;
* `catalog` - каталог данных, которые формируются в результате работы функций трансформациии и используются другими функциями трансформации образуя граф обработки данных;
* `pipeline` - шаги трансформаций.

In [16]:
app = DatapipeApp(ds, catalog, pipeline)

Получившийся граф преобразований:
```
┌─────────────────────────────────────────────────────┐
│           Трансформация: ScanFileList               │
└─────────────────────────────────────────────────────┘
                          │
                          ▼
╔═════════════════════════════════════════════════════╗
║               Данные: cars_scanned                  ║
╚═════════════════════════════════════════════════════╝
```
Транформация `ScanFileList` генерирует информацию об обнаруженных сырых данных `cars_scanned`.

## Вспомогательные функции

Аналог консольной команды `datapipe db create-all`, которая позволяет создать необходимые таблицы для хранения метаданных.

Перед первым запуском datapipe необходимо создать БД и таблицы для меты.

In [17]:
def datapipe__create_db():
  app.ds.meta_dbconn.sqla_metadata.create_all(app.ds.meta_dbconn.con)

Аналог консольной команды `datapipe run`, которая используется для запуска процессинга.

In [18]:
def datapipe__run():
  run_steps(app.ds, app.steps)

Функция для проверки обработанных данных.

Данная функция посточно выводит на экран содержимое обработанных данных.

In [19]:
def check_data(data: str):
  with open(FILEPATH__PROCESSED__CARS.replace("{data}", data), "r") as f:
    for line in f.readlines():
      print(line)

Функция для вывода списка таблиц метаинформации из Data Store.

Данная функция выводит на экран список всех таблиц метаданных.

In [20]:
def check_meta():
  engine = create_engine(f"{sqla_engine}:///{PROJECT_LOCATION}store.sqlite")
  inspector = inspect(engine)
  for table_name in inspector.get_table_names():
    print(table_name)

Функция для вывода содержимого таблицы метаинформации.

Данная функция выводит на экран pandas DataFrame с данными из указанной таблицы меты.

In [21]:
def print_meta(meta_table: str):
  engine = create_engine(f"{sqla_engine}:///{PROJECT_LOCATION}store.sqlite")
  return pd.read_sql_table(meta_table, engine)

## Запуск пайплайна

Для работы Datapipe необходимо хранилище метаданных в виде реляционной БД. В данном примере SQLite будет создана при помощи функции `datapipe__create_db()`. Файлы БД появятся в директории проекта после запуска функции.

In [22]:
datapipe__create_db()

Запуск процессинга.

In [23]:
datapipe__run()

Проверка таблиц меты в БД Data Store после запуска процессинга.

In [24]:
check_meta()

cars_scanned_meta


Функция сканирования после обнаружения сырых данных сгенерировала для каждого файла соответствующую мету:

In [25]:
print_meta("cars_scanned_meta")

,file_name,filepath,size_bytes,hash,create_ts,update_ts,process_ts,delete_ts
0,record_1,data/raw/cars/record_1.json,109,-905267542,1.777051e+09,1.777051e+09,1.777051e+09,NaN
1,record_2,data/raw/cars/record_2.json,109,-2029521359,1.777051e+09,1.777051e+09,1.777051e+09,NaN
2,record_3,data/raw/cars/record_3.json,109,336805490,1.777051e+09,1.777051e+09,1.777051e+09,NaN
3,record_4,data/raw/cars/record_4.json,107,269703770,1.777051e+09,1.777051e+09,1.777051e+09,NaN
4,record_5,data/raw/cars/record_5.json,107,-1311987398,1.777051e+09,1.777051e+09,1.777051e+09,NaN


Также в результате работы функции сканирования появилась не только мета, но и данные, которые в данном случае полностью совпадают с метой.

Важной особенностью является то, что мета используется для отслеживания изменений и построения вычислителнього графа. Так например изменение размера файла `size_bytes` или пути `filepath` сигнализирует об измении в файле, что придёт к пересчёту данных из этого файла, а также ко всем последующим downstream трансформациям. Критерием изменённости является отличие в поле `hash`, которое рассчитывается при каждом запуске и сравнивается с сохранённым ранее значением. Как правило, хэш берётся от всех пользовательских полей меты, что позволяет отслеживать изменения.

Данные же передаются в последующие функции трансформации и могут содержать больший набор полей.

In [26]:
check_data("cars_scanned")

{"file_name":"record_1","filepath":"data\/raw\/cars\/record_1.json","size_bytes":109}

{"file_name":"record_2","filepath":"data\/raw\/cars\/record_2.json","size_bytes":109}

{"file_name":"record_3","filepath":"data\/raw\/cars\/record_3.json","size_bytes":109}

{"file_name":"record_4","filepath":"data\/raw\/cars\/record_4.json","size_bytes":107}

{"file_name":"record_5","filepath":"data\/raw\/cars\/record_5.json","size_bytes":107}



**В результате работы первого шага в пайплайне появилась информация о файлах сырых данных.**

# Шаг 2. Парсинг данных

Сканирование сырых данных позволяет системе описать именование, путь и размер файлов, но не сами данные внутри файлов. Для описания данных и выполнения операций над ними потребуется реализовать разбор данных внутри файлов. На втором шаге процессинга будет происходить разбор - парсинг сырых данных.

## Описание пайплайна

В каталоге метаданных появилась первая запись - `cars_parsed`. Это описание меты для разобранных данных:
* аргумент `store=TableStoreJsonLine` обозначает, что данные будут храниться в формате JSON lines файлов;
* аргумент `filename=FILEPATH__PROCESSED__CARS.format(data="cars_parsed")` определяет шаблон пути для записи обработанных файлов с заполненным именем файла `cars_parsed`;
* аргумент `primary_schema` определяет массив пользовательских полей для меты, которые позволяют связать последующие трансформации с текущей. В данном случае поле `file_name` уникально для каждой записи, полученной из предыдущего шага сканирования, что приведёт к трансформации "один к одному". Остальные поля в схеме описаны с учётом последующих трансформаций.

In [27]:
catalog = Catalog(
    {
        "cars_parsed": Table(
            store=TableStoreJsonLine(
                filename=FILEPATH__PROCESSED__CARS.format(data="cars_parsed"),
                primary_schema=[
                    Column("file_name", String, primary_key=True),
                    Column("model_id", Integer, primary_key=True),
                    Column("manufacture_country", String, primary_key=True),
                    Column("color", String, primary_key=True),
                ],
            )
        ),
    }
)

Далее следуюет описании функции трансформации - это функция, которая будет вызвана в каждой итерации текущей трансформации. Функция принмиает на вход pandas DataFrame с метой сохранённых данных из предыдущего шага трансформации. Также функция должна вернуть мету для обработанных данных.

На данном шаге необходимо прочитать файлы сырых данных, чтобы получить информацию из них. Для этого будет использовано поле `filepath` из меты предыдущей трансформации. После этого информация о каждом значении `file_name` будет обогащена данными из соответствующего файла.

In [28]:
def parse_cars(df__input__cars_scanned: pd.DataFrame) -> pd.DataFrame:
    list_output_dfs = []

    for _, row in df__input__cars_scanned.iterrows():
        with open(row["filepath"], "r") as file:
            json_record = json.load(file)

        df__record_parsed = pd.DataFrame(
            {
                "file_name": row["file_name"],
                "model_id": json_record["model_id"],
                "manufacture_country": json_record["manufacture_country"],
                "color": json_record["color"],
                "price": json_record["price"],
                "year": json_record["year"],
                "new": json_record["new"],
            },
            index=[0]
        )

        list_output_dfs.append(df__record_parsed)

    df__output__cars_parsed = pd.concat(list_output_dfs)

    df__output__cars_parsed["new"] = df__output__cars_parsed["new"].astype(int)

    return df__output__cars_parsed

В списке шагов пайплайна появилась новая транформация с функцией `parse_cars` и следующими аргументами:
* `inputs=["cars_scanned"],` - входная мета из предыдущей трансформации;
* `outputs=["cars_parsed"],` - выходная мета, которая будет получена в результате работы текущей функции трансформации;
* `kwargs={},` - дополнительные аргументы для функции трансформации `parse_cars()`, в данном случае аргументы не используются;
* `chunk_size=1,` - размер чанка для одной итерации шага. Единица обозначает, что за раз будет передан только один уникальный `file_name`;
* `executor_config=ExecutorConfig(parallelism=1),` - настройки запуска транформации. В данном случае за раз функция транформации будет запущена в 1 поток;
* `transform_keys=["file_name"],` - набор ключей транформации, уникальные значения которых будут обрабатываться независимо друг от друга. В данном случае будут последовательно переданы именования файлов `file_name` из меты предыдущей трансформации.

In [29]:
pipeline = Pipeline(
    [
        ScanFileList(
            filename_pattern=FILEPATH__RAW__CARS,
            filename_output=FILEPATH__PROCESSED__CARS,
            output="cars_scanned",
            labels=[
                ("entity", "cars"),
                ("layer", "scan"),
                ("environment", "prod"),
            ],
        ),
        BatchTransform(
            parse_cars,
            inputs=["cars_scanned"],
            outputs=["cars_parsed"],
            kwargs={},
            chunk_size=1,
            executor_config=ExecutorConfig(parallelism=1),
            transform_keys=[
                "file_name",
            ],
            labels=[
                ("entity", "cars"),
                ("layer", "parse"),
                ("environment", "prod"),
            ],
        ),
    ]
)

Повторная инициализация пайплайна с обновлёнными аргументами.

In [30]:
app = DatapipeApp(ds, catalog, pipeline)

Обновлённый граф преобразований:
```
┌─────────────────────────────────────────────────────┐
│           Трансформация: ScanFileList               │
└─────────────────────────────────────────────────────┘
                          │
                          ▼
╔═════════════════════════════════════════════════════╗
║               Данные: cars_scanned                  ║
╚═════════════════════════════════════════════════════╝
                          │
                          ▼
┌─────────────────────────────────────────────────────┐
│              Трансформация: parse_cars              │  ◄ новая транформация
└─────────────────────────────────────────────────────┘
                          │
                          ▼
╔═════════════════════════════════════════════════════╗
║               Данные: cars_parsed                   ║  ◄ новые данные
╚═════════════════════════════════════════════════════╝
```
Мета данных `cars_scanned` теперь используются трансформацией `parse_cars`, в результате работы которой появляются данные и мета `cars_parsed`.

## Запуск пайплайна

Обновление таблиц для хранения меты.

In [31]:
datapipe__create_db()

Появились две новые таблицы меты:
* `cars_parsed_meta` - таблица из каталога для меты данных, посчитанных на текущем шаге;
* `parse_cars_61dfc936e9_meta` - таблица для меты шага трансформации, описывает результат работы функции трансформации;

In [32]:
check_meta()

cars_parsed_meta
cars_scanned_meta
parse_cars_61dfc936e9_meta


Таблицы меты созданы, но ещё не заполнены.

In [33]:
print_meta("parse_cars_61dfc936e9_meta")

,file_name,process_ts,is_success,priority,error


In [34]:
print_meta("cars_parsed_meta")

,file_name,model_id,manufacture_country,color,hash,create_ts,update_ts,process_ts,delete_ts


Запуск процессинга обрабатывает пять записей за пять итераций.

In [35]:
datapipe__run()

  0%|          | 0/5 [00:00<?, ?it/s]

Мета трансформации использует ключи трансформации для идентификации каждой итерации запуска шага. В данном случае был указан один ключ трансформации `file_name`.

Путь уникальных `file_name` обрабатываются каждый отдельно, что приводит к пяти итерациям выше.

In [36]:
print_meta("parse_cars_61dfc936e9_meta")

,file_name,process_ts,is_success,priority,error
0,record_1,1.777051e+09,True,0,NaN
1,record_2,1.777051e+09,True,0,NaN
2,record_3,1.777051e+09,True,0,NaN
3,record_4,1.777051e+09,True,0,NaN
4,record_5,1.777051e+09,True,0,NaN


Мета данных описывает все поля, указанные в каталоге данных. Для текущего шага были описаны следующие поля:
1. `file_name`
2. `model_id`
3. `manufacture_country`
4. `color`


In [37]:
print_meta("cars_parsed_meta")

,file_name,model_id,manufacture_country,color,hash,create_ts,update_ts,process_ts,delete_ts
0,record_1,1,Germany,Red,-815812175,1.777051e+09,1.777051e+09,1.777051e+09,NaN
1,record_2,1,Germany,Blue,2133128644,1.777051e+09,1.777051e+09,1.777051e+09,NaN
2,record_3,3,Germany,Blue,253623355,1.777051e+09,1.777051e+09,1.777051e+09,NaN
3,record_4,4,Japan,Red,1504749412,1.777051e+09,1.777051e+09,1.777051e+09,NaN
4,record_5,5,Japan,Blue,-582289655,1.777051e+09,1.777051e+09,1.777051e+09,NaN


Помимо меты также были записаны данные по указанному в каталоге шаблону пути. В отличие от первого шага, теперь данные отличаются от меты - данные содержат больше полей.

Поля `price`, `year`, `new` сохранены только в данных, так как полей `file_name`, `model_id`, `manufacture_country`, `color` достаточно для уникальной идентификации каждой записи и для расчёта индекса транформации для последующих шагов.

---
> **Важно!**
---
> В мету желательно сохранять минимально необходимый набор полей, так как увеличение меты замедляет построение графа вычислений.
---

In [38]:
check_data("cars_parsed")

{"file_name":"record_1","model_id":1,"manufacture_country":"Germany","color":"Red","price":15000,"year":2023,"new":0}

{"file_name":"record_2","model_id":1,"manufacture_country":"Germany","color":"Blue","price":20000,"year":2026,"new":1}

{"file_name":"record_3","model_id":3,"manufacture_country":"Germany","color":"Blue","price":25000,"year":2026,"new":1}

{"file_name":"record_4","model_id":4,"manufacture_country":"Japan","color":"Red","price":20000,"year":2024,"new":0}

{"file_name":"record_5","model_id":5,"manufacture_country":"Japan","color":"Blue","price":15000,"year":2025,"new":1}



**В результате работы второго шага был произведён разбор сырых данных.**

# Шаг 3. Price by Manufacture Country

Теперь разобранные данные можно использовать для расчёта необходимых бизнесу метрик. На данном шаге будет продемонстрирован расчёт метрики "Суммарная стоимость авто по странам производства".

## Описание пайплайна

В каталоге данных появилась новая запись - `price_by_manufacture_country`. Это описание меты для новой метрики.

Примечательно то, что в новой записи фигурирет только одно поле `manufacture_country`. Поскольку метрика расчитывается как суммарная стоимость для каждой страны, то в результате подсчёта останутся только уникальные именования стран, которые пользолят однозначно идентифицировать каждую запись. Иными словами, единственным первичным ключом в данных будет именование страны - поле `manufacture_country`.

In [39]:
catalog = Catalog(
    {
        "cars_parsed": Table(
            store=TableStoreJsonLine(
                filename=FILEPATH__PROCESSED__CARS.format(data="cars_parsed"),
                primary_schema=[
                    Column("file_name", String, primary_key=True),
                    Column("model_id", Integer, primary_key=True),
                    Column("manufacture_country", String, primary_key=True),
                    Column("color", String, primary_key=True),
                ],
            )
        ),
        "price_by_manufacture_country": Table(
            store=TableStoreJsonLine(
                filename=FILEPATH__PROCESSED__CARS.format(data="price_by_manufacture_country"),
                primary_schema=[
                    Column("manufacture_country", String, primary_key=True),
                ],
            )
        ),
    }
)

Функция трансформации `agg__price_by_manufacture_country` будет принимать данные из `df__input__cars_parsed` и использовать поля `manufacture_country` и `price` для расчёта метрики, а затем возвращать посчитанные данные `df__output__price_by_manufacture_country`.

In [40]:
def agg__price_by_manufacture_country(
        df__input__cars_parsed: pd.DataFrame,
) -> pd.DataFrame:
    df__output__price_by_manufacture_country = df__input__cars_parsed[["manufacture_country", "price"]]
    df__output__price_by_manufacture_country = (
        df__output__price_by_manufacture_country
        .groupby("manufacture_country", as_index=False)
        .sum()
    )

    return df__output__price_by_manufacture_country

В списке шагов пайплайна появился новый шаг `BatchTransform` с вышеописанной функцией трансформации `agg__price_by_manufacture_country`.

Набор ключей трансформации состоит из одного поля `manufacture_country`. Это значит, что итерирование будет происходить по уникальным значениям поля `manufacture_country`. Входная мета `cars_parsed` содержит следующие значения данного поля:
1. `Germany`
2. `Germany`
3. `Germany`
4. `Japan`
5. `Japan`

Таким образом, функция транформации будет запущена два раза. Сперва для всех данных `Germany`, затем для всех данных `Japan`.

In [41]:
pipeline = Pipeline(
    [
        ScanFileList(
            filename_pattern=FILEPATH__RAW__CARS,
            filename_output=FILEPATH__PROCESSED__CARS,
            output="cars_scanned",
            labels=[
                ("entity", "cars"),
                ("layer", "scan"),
                ("environment", "prod"),
            ],
        ),
        BatchTransform(
            parse_cars,
            inputs=["cars_scanned"],
            outputs=["cars_parsed"],
            kwargs={},
            chunk_size=1,
            executor_config=ExecutorConfig(parallelism=1),
            transform_keys=[
                "file_name",
            ],
            labels=[
                ("entity", "cars"),
                ("layer", "parse"),
                ("environment", "prod"),
            ],
        ),
        BatchTransform(
            agg__price_by_manufacture_country,
            inputs=["cars_parsed"],
            outputs=["price_by_manufacture_country"],
            kwargs={},
            chunk_size=1,
            executor_config=ExecutorConfig(parallelism=1),
            transform_keys=[
                "manufacture_country",
            ],
            labels=[
                ("entity", "cars"),
                ("layer", "agg"),
                ("environment", "prod"),
            ],
        ),
    ]
)

Повторная инициализация пайплайна с обновлёнными аргументами.

In [42]:
app = DatapipeApp(ds, catalog, pipeline)

Обновлённый граф преобразований:
```
┌─────────────────────────────────────────────────────┐
│           Трансформация: ScanFileList               │
└─────────────────────────────────────────────────────┘
                          │
                          ▼
╔═════════════════════════════════════════════════════╗
║               Данные: cars_scanned                  ║
╚═════════════════════════════════════════════════════╝
                          │
                          ▼
┌─────────────────────────────────────────────────────┐
│              Трансформация: parse_cars              │
└─────────────────────────────────────────────────────┘
                          │
                          ▼
╔═════════════════════════════════════════════════════╗
║               Данные: cars_parsed                   ║
╚═════════════════════════════════════════════════════╝
                          │
                          ▼
┌─────────────────────────────────────────────────────┐
│    Трансформация: agg__price_by_manufacture_country │  ◄ новая трансформация
└─────────────────────────────────────────────────────┘
                          │
                          ▼
╔═════════════════════════════════════════════════════╗
║       Данные: price_by_manufacture_country          ║  ◄ новые данные
╚═════════════════════════════════════════════════════╝
```
Трансформация `agg__price_by_manufacture_country` использует мету данных `cars_parsed` и рассчитывает данные `price_by_manufacture_country`.

## Запуск пайплайна

Создание таблиц для новой меты.

In [43]:
datapipe__create_db()

Были созданы две новые таблицы для меты:
* `agg__price_by_manufacture_country_c1e1b9beec_meta` - мета функции трансформации `agg__price_by_manufacture_country`;
* `price_by_manufacture_country_meta` - мета для данных `price_by_manufacture_country`.

In [44]:
check_meta()

agg__price_by_manufacture_country_c1e1b9beec_meta
cars_parsed_meta
cars_scanned_meta
parse_cars_61dfc936e9_meta
price_by_manufacture_country_meta


Таблицы с новой метой пока пустые.

In [45]:
print_meta("agg__price_by_manufacture_country_c1e1b9beec_meta")

,manufacture_country,process_ts,is_success,priority,error


In [46]:
print_meta("price_by_manufacture_country_meta")

,manufacture_country,hash,create_ts,update_ts,process_ts,delete_ts


Запуск происходит в две итерации: сперва для всех данных `Germany`, затем для всех данных `Japan`.

In [47]:
datapipe__run()

  0%|          | 0/2 [00:00<?, ?it/s]

Выходная мета содержит уникальные значения ключей трансформации.

In [48]:
print_meta("agg__price_by_manufacture_country_c1e1b9beec_meta")

,manufacture_country,process_ts,is_success,priority,error
0,Germany,1.777051e+09,True,0,NaN
1,Japan,1.777051e+09,True,0,NaN


In [49]:
print_meta("price_by_manufacture_country_meta")

,manufacture_country,hash,create_ts,update_ts,process_ts,delete_ts
0,Germany,1060410227,1.777051e+09,1.777051e+09,1.777051e+09,NaN
1,Japan,-1599460678,1.777051e+09,1.777051e+09,1.777051e+09,NaN


Значение посчитанной метрики находится только в данных, а мета лишь описывает существование соответствующих записей.

Также мета позволит выборочно пересчитать метрики только для тех стран, в сырых данных которых произошли изменения.

In [50]:
check_data("price_by_manufacture_country")

{"manufacture_country":"Germany","price":60000}

{"manufacture_country":"Japan","price":35000}



**В результате работы третьего шага на основе разобранных данных была посчитана метрика "Суммарная стоимость авто по странам производства".**

# Шаг 4. Price by Color

Выполним расчёт ещё одной метрики "Суммарная стоимость авто по цвету авто".

## Описание пайплайна

Аналогично предыдущему шагу, в каталоге появилась нвоая запись `price_by_color` с полем `color`, так как теперь агрегация будет выполнена по цвету авто.

In [51]:
catalog = Catalog(
    {
        "cars_parsed": Table(
            store=TableStoreJsonLine(
                filename=FILEPATH__PROCESSED__CARS.format(data="cars_parsed"),
                primary_schema=[
                    Column("file_name", String, primary_key=True),
                    Column("model_id", Integer, primary_key=True),
                    Column("manufacture_country", String, primary_key=True),
                    Column("color", String, primary_key=True),
                ],
            )
        ),
        "price_by_manufacture_country": Table(
            store=TableStoreJsonLine(
                filename=FILEPATH__PROCESSED__CARS.format(data="price_by_manufacture_country"),
                primary_schema=[
                    Column("manufacture_country", String, primary_key=True),
                ],
            )
        ),
        "price_by_color": Table(
            store=TableStoreJsonLine(
                filename=FILEPATH__PROCESSED__CARS.format(data="price_by_color"),
                primary_schema=[
                    Column("color", String, primary_key=True),
                ],
            )
        ),
    }
)

Функция трансформации теперь использует группировку по столбцу `color`.

In [52]:
def agg__price_by_color(
        df__input__cars_parsed: pd.DataFrame,
) -> pd.DataFrame:
    df__output__price_by_color = df__input__cars_parsed[["color", "price"]]
    df__output__price_by_color = (
        df__output__price_by_color
        .groupby("color", as_index=False)
        .sum()
    )

    return df__output__price_by_color

Также был добавлен новый шаг `BatchTransform` с функций транформации `agg__price_by_color`, для которой указан ключ трансформации `color`.

In [53]:
pipeline = Pipeline(
    [
        ScanFileList(
            filename_pattern=FILEPATH__RAW__CARS,
            filename_output=FILEPATH__PROCESSED__CARS,
            output="cars_scanned",
            labels=[
                ("entity", "cars"),
                ("layer", "scan"),
                ("environment", "prod"),
            ],
        ),
        BatchTransform(
            parse_cars,
            inputs=["cars_scanned"],
            outputs=["cars_parsed"],
            kwargs={},
            chunk_size=1,
            executor_config=ExecutorConfig(parallelism=1),
            transform_keys=[
                "file_name",
            ],
            labels=[
                ("entity", "cars"),
                ("layer", "parse"),
                ("environment", "prod"),
            ],
        ),
        BatchTransform(
            agg__price_by_manufacture_country,
            inputs=["cars_parsed"],
            outputs=["price_by_manufacture_country"],
            kwargs={},
            chunk_size=1,
            executor_config=ExecutorConfig(parallelism=1),
            transform_keys=[
                "manufacture_country",
            ],
            labels=[
                ("entity", "cars"),
                ("layer", "agg"),
                ("environment", "prod"),
            ],
        ),
        BatchTransform(
            agg__price_by_color,
            inputs=["cars_parsed"],
            outputs=["price_by_color"],
            kwargs={},
            chunk_size=1,
            executor_config=ExecutorConfig(parallelism=1),
            transform_keys=[
                "color",
            ],
            labels=[
                ("entity", "cars"),
                ("layer", "agg"),
                ("environment", "prod"),
            ],
        ),
    ]
)

Повторная инициализация пайплайна с обновлёнными аргументами.

In [54]:
app = DatapipeApp(ds, catalog, pipeline)

Обновлённый граф преобразований:
```
                    ┌─────────────────────────────────────────────────────┐
                    │             Трансформация: ScanFileList             │
                    └─────────────────────────────────────────────────────┘
                                               │
                                               ▼
                    ╔═════════════════════════════════════════════════════╗
                    ║                Данные: cars_scanned                 ║
                    ╚═════════════════════════════════════════════════════╝
                                               │
                                               ▼
                    ┌─────────────────────────────────────────────────────┐
                    │              Трансформация: parse_cars              │
                    └─────────────────────────────────────────────────────┘
                                               │
                                               ▼
                    ╔═════════════════════════════════════════════════════╗
                    ║                 Данные: cars_parsed                 ║
                    ╚═════════════════════════════════════════════════════╝
                                               │
                       ┌───────────────────────┴────────────────────────┐
                       │                                                │
                       ▼                                                ▼
┌─────────────────────────────────────────────┐  ┌─────────────────────────────────────────────┐
│               Трансформация:                │  │               Трансформация:                │  ◄ новая трансформация
│      agg__price_by_manufacture_country      │  │             agg__price_by_color             │
└─────────────────────────────────────────────┘  └─────────────────────────────────────────────┘
                       │                                                │
                       ▼                                                ▼
╔═════════════════════════════════════════════╗  ╔═════════════════════════════════════════════╗
║                   Данные:                   ║  ║                   Данные:                   ║  ◄ новые данные
║        price_by_manufacture_country         ║  ║               price_by_color                ║
╚═════════════════════════════════════════════╝  ╚═════════════════════════════════════════════╝
```
Теперь данные `cars_parsed` используются сразу двумя транформациями:
* трансформацией `agg__price_by_manufacture_country` из предыдущего шага;
* новой трансформацией `agg__price_by_color`.

Результатом работы трансформации `agg__price_by_color` являются данные и мета `price_by_color`.

## Запуск пайплайна

Создание таблиц для новой меты.

In [55]:
datapipe__create_db()

Были созданы две новые таблицы для меты:
* `agg__price_by_color_99200df109_meta` - мета функции трансформации `agg__price_by_color`;
* `price_by_color_meta` - мета для данных `price_by_color`.

In [56]:
check_meta()

agg__price_by_color_99200df109_meta
agg__price_by_manufacture_country_c1e1b9beec_meta
cars_parsed_meta
cars_scanned_meta
parse_cars_61dfc936e9_meta
price_by_color_meta
price_by_manufacture_country_meta


Таблицы с новой метой пока пустые.

In [57]:
print_meta("agg__price_by_color_99200df109_meta")

,color,process_ts,is_success,priority,error


In [58]:
print_meta("price_by_color_meta")

,color,hash,create_ts,update_ts,process_ts,delete_ts


Запуск происходит в две итерации: сперва для всех данных `Blue`, затем для всех данных `Red`.

In [59]:
datapipe__run()

  0%|          | 0/2 [00:00<?, ?it/s]

Выходная мета содержит уникальные значения ключей трансформации.

In [60]:
print_meta("agg__price_by_color_99200df109_meta")

,color,process_ts,is_success,priority,error
0,Blue,1.777051e+09,True,0,NaN
1,Red,1.777051e+09,True,0,NaN


In [61]:
print_meta("price_by_color_meta")

,color,hash,create_ts,update_ts,process_ts,delete_ts
0,Blue,-113662495,1.777051e+09,1.777051e+09,1.777051e+09,NaN
1,Red,442594030,1.777051e+09,1.777051e+09,1.777051e+09,NaN


Значения посчитанной метрики находятся в данных.

In [62]:
check_data("price_by_color")

{"color":"Blue","price":60000}

{"color":"Red","price":35000}



**В результате работы четвёртого шага на основе разобранных данных была посчитана метрика "Суммарная стоимость авто по цвету авто".**

# Шаг 5. Price by Manufacture Country and Color

Теперь рассмотрим чуть более сложную метрику "Суммарная стоимость авто по странам производства и по цвету авто".

## Описание пайплайна

В каталоге данных появилась запись `price_by_manufacture_country_and_color`, которая имеет два поля:
* `manufacture_country` - страна производства;
* `color` - цвет авто.

Оба поля будут использоваться при расчёте новой метрики.

In [63]:
catalog = Catalog(
    {
        "cars_parsed": Table(
            store=TableStoreJsonLine(
                filename=FILEPATH__PROCESSED__CARS.format(data="cars_parsed"),
                primary_schema=[
                    Column("file_name", String, primary_key=True),
                    Column("model_id", Integer, primary_key=True),
                    Column("manufacture_country", String, primary_key=True),
                    Column("color", String, primary_key=True),
                ],
            )
        ),
        "price_by_manufacture_country": Table(
            store=TableStoreJsonLine(
                filename=FILEPATH__PROCESSED__CARS.format(data="price_by_manufacture_country"),
                primary_schema=[
                    Column("manufacture_country", String, primary_key=True),
                ],
            )
        ),
        "price_by_color": Table(
            store=TableStoreJsonLine(
                filename=FILEPATH__PROCESSED__CARS.format(data="price_by_color"),
                primary_schema=[
                    Column("color", String, primary_key=True),
                ],
            )
        ),
        "price_by_manufacture_country_and_color": Table(
            store=TableStoreJsonLine(
                filename=FILEPATH__PROCESSED__CARS.format(data="price_by_manufacture_country_and_color"),
                primary_schema=[
                    Column("manufacture_country", String, primary_key=True),
                    Column("color", String, primary_key=True),
                ],
            )
        ),
    }
)

Аггрегация теперь также производится по двум полям.

In [64]:
def agg__price_by_manufacture_country_and_color(
        df__input__cars_parsed: pd.DataFrame,
) -> pd.DataFrame:
    df__output__price_by_manufacture_country_and_color = df__input__cars_parsed[["manufacture_country", "color", "price"]]
    df__output__price_by_manufacture_country_and_color = (
        df__output__price_by_manufacture_country_and_color
        .groupby(["manufacture_country", "color"], as_index=False)
        .sum()
    )

    return df__output__price_by_manufacture_country_and_color

Новый шаг `BatchTransform` с функцией трансформации `agg__price_by_manufacture_country_and_color` использует оба поля в качестве ключей трансформации: `manufacture_country` и `color`.

In [65]:
pipeline = Pipeline(
    [
        ScanFileList(
            filename_pattern=FILEPATH__RAW__CARS,
            filename_output=FILEPATH__PROCESSED__CARS,
            output="cars_scanned",
            labels=[
                ("entity", "cars"),
                ("layer", "scan"),
                ("environment", "prod"),
            ],
        ),
        BatchTransform(
            parse_cars,
            inputs=["cars_scanned"],
            outputs=["cars_parsed"],
            kwargs={},
            chunk_size=1,
            executor_config=ExecutorConfig(parallelism=1),
            transform_keys=[
                "file_name",
            ],
            labels=[
                ("entity", "cars"),
                ("layer", "parse"),
                ("environment", "prod"),
            ],
        ),
        BatchTransform(
            agg__price_by_manufacture_country,
            inputs=["cars_parsed"],
            outputs=["price_by_manufacture_country"],
            kwargs={},
            chunk_size=1,
            executor_config=ExecutorConfig(parallelism=1),
            transform_keys=[
                "manufacture_country",
            ],
            labels=[
                ("entity", "cars"),
                ("layer", "agg"),
                ("environment", "prod"),
            ],
        ),
        BatchTransform(
            agg__price_by_color,
            inputs=["cars_parsed"],
            outputs=["price_by_color"],
            kwargs={},
            chunk_size=1,
            executor_config=ExecutorConfig(parallelism=1),
            transform_keys=[
                "color",
            ],
            labels=[
                ("entity", "cars"),
                ("layer", "agg"),
                ("environment", "prod"),
            ],
        ),
        BatchTransform(
            agg__price_by_manufacture_country_and_color,
            inputs=["cars_parsed"],
            outputs=["price_by_manufacture_country_and_color"],
            kwargs={},
            chunk_size=1,
            executor_config=ExecutorConfig(parallelism=1),
            transform_keys=[
                "manufacture_country",
                "color",
            ],
            labels=[
                ("entity", "cars"),
                ("layer", "agg"),
                ("environment", "prod"),
            ],
        ),
    ]
)

Повторная инициализация пайплайна с обновлёнными аргументами.

In [66]:
app = DatapipeApp(ds, catalog, pipeline)

Обновлённый граф преобразований:
```
                                             ┌─────────────────────────────────────────────────────┐
                                             │             Трансформация: ScanFileList             │
                                             └─────────────────────────────────────────────────────┘
                                                                        │
                                                                        ▼
                                             ╔═════════════════════════════════════════════════════╗
                                             ║                Данные: cars_scanned                 ║
                                             ╚═════════════════════════════════════════════════════╝
                                                                        │
                                                                        ▼
                                             ┌─────────────────────────────────────────────────────┐
                                             │              Трансформация: parse_cars              │
                                             └─────────────────────────────────────────────────────┘
                                                                        │
                                                                        ▼
                                             ╔═════════════════════════════════════════════════════╗
                                             ║                 Данные: cars_parsed                 ║
                                             ╚═════════════════════════════════════════════════════╝
                                                                        │
                       ┌────────────────────────────────────────────────┼────────────────────────────────────────────────┐
                       │                                                │                                                │
                       ▼                                                ▼                                                ▼
┌─────────────────────────────────────────────┐  ┌─────────────────────────────────────────────┐  ┌─────────────────────────────────────────────┐
│               Трансформация:                │  │               Трансформация:                │  │               Трансформация:                │  ◄ новая трансформация
│      agg__price_by_manufacture_country      │  │             agg__price_by_color             │  │ agg__price_by_manufacture_country_and_color │
└─────────────────────────────────────────────┘  └─────────────────────────────────────────────┘  └─────────────────────────────────────────────┘
                       │                                                │                                                │
                       ▼                                                ▼                                                ▼
╔═════════════════════════════════════════════╗  ╔═════════════════════════════════════════════╗  ╔═════════════════════════════════════════════╗
║                   Данные:                   ║  ║                   Данные:                   ║  ║                   Данные:                   ║  ◄ новые данные
║        price_by_manufacture_country         ║  ║               price_by_color                ║  ║   price_by_manufacture_country_and_color    ║
╚═════════════════════════════════════════════╝  ╚═════════════════════════════════════════════╝  ╚═════════════════════════════════════════════╝
```
Теперь данные `cars_parsed` используются ещё одной функцией трансформации `agg__price_by_manufacture_country_and_color`.

## Запуск пайплайна

Создание таблиц для новой меты.

In [67]:
datapipe__create_db()

Были созданы две новые таблицы для меты:
* `agg__price_by_manufacture_country_and_color_5b897d7adc_meta` - мета функции трансформации `agg__price_by_manufacture_country_and_color`;
* `price_by_manufacture_country_and_color_meta` - мета для данных `price_by_manufacture_country_and_color`.

In [68]:
check_meta()

agg__price_by_color_99200df109_meta
agg__price_by_manufacture_country_and_color_5b897d7adc_meta
agg__price_by_manufacture_country_c1e1b9beec_meta
cars_parsed_meta
cars_scanned_meta
parse_cars_61dfc936e9_meta
price_by_color_meta
price_by_manufacture_country_and_color_meta
price_by_manufacture_country_meta


Таблицы с новой метой пока пустые.

Теперь в таблицах меты фигурируют оба поля: `manufacture_country` и `color`.

In [69]:
print_meta("agg__price_by_manufacture_country_and_color_5b897d7adc_meta")

,manufacture_country,color,process_ts,is_success,priority,error


In [70]:
print_meta("price_by_manufacture_country_and_color_meta")

,manufacture_country,color,hash,create_ts,update_ts,process_ts,delete_ts


Теперь на перебор всех уникальных ключей трансформации требукется 4 итерации:
1. `Germany` - `Blue`.
2. `Germany` - `Red`.
3. `Japan` - `Blue`.
4. `Japan` - `Red`.

In [71]:
datapipe__run()

  0%|          | 0/4 [00:00<?, ?it/s]

В посчитанной мете фигурируют вышеописанные уникальные комбинации ключей трансформации.

In [72]:
print_meta("agg__price_by_manufacture_country_and_color_5b897d7adc_meta")

,manufacture_country,color,process_ts,is_success,priority,error
0,Germany,Blue,1.777051e+09,True,0,NaN
1,Germany,Red,1.777051e+09,True,0,NaN
2,Japan,Blue,1.777051e+09,True,0,NaN
3,Japan,Red,1.777051e+09,True,0,NaN


In [73]:
print_meta("price_by_manufacture_country_and_color_meta")

,manufacture_country,color,hash,create_ts,update_ts,process_ts,delete_ts
0,Germany,Blue,396556664,1.777051e+09,1.777051e+09,1.777051e+09,NaN
1,Germany,Red,-272202745,1.777051e+09,1.777051e+09,1.777051e+09,NaN
2,Japan,Blue,815985979,1.777051e+09,1.777051e+09,1.777051e+09,NaN
3,Japan,Red,965357621,1.777051e+09,1.777051e+09,1.777051e+09,NaN


Аналогично предыдущим шагам, в данных сохранены значения метрики для каждой комбинации ключей трансформации.

In [74]:
check_data("price_by_manufacture_country_and_color")

{"manufacture_country":"Germany","color":"Blue","price":45000}

{"manufacture_country":"Germany","color":"Red","price":15000}

{"manufacture_country":"Japan","color":"Blue","price":15000}

{"manufacture_country":"Japan","color":"Red","price":20000}



**В результате работы пятого шага на основе разобранных данных была посчитана метрика "Суммарная стоимость авто по странам производства и по цвету авто".**

# Работа с метаданными - выводы

TO DO

# Очистка данных

Очистка сгенерированных данных для повторного запуска процессинга.

In [75]:
if os.path.exists(f"{PROJECT_LOCATION}data/"):
  shutil.rmtree(f"{PROJECT_LOCATION}data/")

for file_to_clean in FILES_TO_CLEAN:
  if os.path.exists(f"{PROJECT_LOCATION}{file_to_clean}"):
    os.remove(f"{PROJECT_LOCATION}{file_to_clean}")